In [8]:
import os
import joblib
import pandas as pd
import numpy as np
import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tabpfn import TabPFNRegressor
import torch
import optuna
import sys

optuna.logging.set_verbosity(optuna.logging.WARNING)

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))

### Creazione e addestramento dei modelli

In [9]:
class TabPFNQuantileWrapper:
    def __init__(self, fitted_model, quantile):
        self.fitted_model = fitted_model
        self.quantile = quantile
        
    def predict(self, X):
        # Alcune versioni di TabPFN accettano il parametro quantile direttamente.
        # Gestiamo robustamente l'estrazione dei quantili dal modello probabilistico.
        try:
            return self.fitted_model.predict(X, quantile=self.quantile)
        except TypeError:
            try:
                # Fallback se accetta una lista di quantili
                preds = self.fitted_model.predict(X, quantiles=[0.1, 0.5, 0.9])
                if self.quantile == 0.1: return preds[:, 0]
                elif self.quantile == 0.5: return preds[:, 1]
                elif self.quantile == 0.9: return preds[:, 2]
            except Exception:
                # Fallback matematico basato sulla media e deviazione standard previste da TabPFN
                mean, std = self.fitted_model.predict(X, return_std=True)
                if self.quantile == 0.1:
                    return mean - 1.2816 * std
                elif self.quantile == 0.5:
                    return mean
                elif self.quantile == 0.9:
                    return mean + 1.2816 * std

In [10]:
def load_and_split_data(train_path, test_path, target_col='target_assignments'):
    """Carica i dati encodati, separa feature e target, e preserva gli ID."""
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    test_ids = df_test['operator_id']
    
    X_train = df_train.drop(columns=[target_col, 'operator_id'])
    y_train = df_train[target_col]
    
    X_test = df_test.drop(columns=[target_col, 'operator_id'])
    y_test = df_test[target_col]
    
    print(f"Dataset caricati: Train ({X_train.shape[0]} righe), Test ({X_test.shape[0]} righe)")
    return X_train, y_train, X_test, y_test, test_ids

X_train, y_train, X_test, y_test, test_ids = load_and_split_data('data/train_dataset.csv', 'data/test_dataset.csv')

Dataset caricati: Train (1738 righe), Test (435 righe)


In [11]:
def prepare_asp_facts(X_test, y_test, q10, q50, q90):
    """
    Combina le predizioni con i dati di contesto e calcola lo score di incertezza.
    """
    results_df = X_test.copy()
    
    # La predizione da suggerire all'ASP arrotondata all'intero più vicino
    results_df['predicted_assignments'] = np.round(q50).astype(int)
    
    # Valore reale
    results_df['actual_assignments'] = y_test.values
    
    # Calcolo dell'Incertezza
    # Più il valore è alto, maggiore è l'incertezza del modello.
    results_df['uncertainty_score'] = q90 - q10
    
    # Estrazione del burdenScore
    burden_col = [col for col in X_test.columns if 'burdenScore' in col][0]
    
    print("\nAnteprima dei dati pronti per essere convertiti in fatti ASP:")
    display_cols = ['predicted_assignments', 'uncertainty_score', burden_col, 'density_ratio']
    display(results_df[display_cols].head())
    
    return results_df



In [12]:
def train_xgboost_quantiles(X_train, y_train, X_test, learning_rate=0.05, max_depth=5, n_estimators=100):
    q10_model = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.1, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    q50_model = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.5, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    q90_model = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.9, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    
    q10_model.fit(X_train, y_train)
    q50_model.fit(X_train, y_train)
    q90_model.fit(X_train, y_train)
    
    return (q10_model, q50_model, q90_model), q10_model.predict(X_test), q50_model.predict(X_test), q90_model.predict(X_test)


def train_lightgbm_quantiles(X_train, y_train, X_test, learning_rate=0.05, num_leaves=31, n_estimators=100):
    q10_model = LGBMRegressor(objective='quantile', alpha=0.1, learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators, random_state=42, verbose=-1)
    q50_model = LGBMRegressor(objective='quantile', alpha=0.5, learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators, random_state=42, verbose=-1)
    q90_model = LGBMRegressor(objective='quantile', alpha=0.9, learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators, random_state=42, verbose=-1)
    
    q10_model.fit(X_train, y_train)
    q50_model.fit(X_train, y_train)
    q90_model.fit(X_train, y_train)
    
    return (q10_model, q50_model, q90_model), q10_model.predict(X_test), q50_model.predict(X_test), q90_model.predict(X_test)


def train_sklearn_gb_quantiles(X_train, y_train, X_test, learning_rate=0.05, max_depth=5, n_estimators=100):
    q10_model = GradientBoostingRegressor(loss='quantile', alpha=0.1, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    q50_model = GradientBoostingRegressor(loss='quantile', alpha=0.5, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    q90_model = GradientBoostingRegressor(loss='quantile', alpha=0.9, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    
    q10_model.fit(X_train, y_train)
    q50_model.fit(X_train, y_train)
    q90_model.fit(X_train, y_train)
    
    return (q10_model, q50_model, q90_model), q10_model.predict(X_test), q50_model.predict(X_test), q90_model.predict(X_test)

def train_tabpfn_quantiles(X_train, y_train, X_test, n_ensemble=4, device='cpu'):
    # In TabPFN fittiamo una volta sola poiché calcola l'intera distribuzione a posteriori!
    
    try:
        # Prova con la sintassi per TabPFN v2.x+
        base_model = TabPFNRegressor(n_estimators=n_ensemble, device=device)
    except TypeError:
        # Fallback per TabPFN v0.x / v1.x
        base_model = TabPFNRegressor(N_ensemble_configurations=n_ensemble, device=device)

    base_model.fit(X_train, y_train)
    
    # Istanziamo i tre wrapper compatibili
    q10_wrapper = TabPFNQuantileWrapper(base_model, 0.1)
    q50_wrapper = TabPFNQuantileWrapper(base_model, 0.5)
    q90_wrapper = TabPFNQuantileWrapper(base_model, 0.9)
    
    # Generiamo i vettori di predizione richiesti
    q10_pred = q10_wrapper.predict(X_test)
    q50_pred = q50_wrapper.predict(X_test)
    q90_pred = q90_wrapper.predict(X_test)
    
    return (q10_wrapper, q50_wrapper, q90_wrapper), q10_pred, q50_pred, q90_pred

In [13]:
def evaluate_model(model_name, y_test, q50, q10, q90):
    mae = mean_absolute_error(y_test, q50)
    rmse = np.sqrt(mean_squared_error(y_test, q50))
    r2 = r2_score(y_test, q50)
    coverage = np.mean((y_test >= q10) & (y_test <= q90))

    print(f"{model_name:<12} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R^2: {r2:.3f} | Copertura: {coverage*100:.1f}%")
    return mae

### Ottimizzazione Hyperparametri con Optuna

In [14]:
def objective_xgb(trial):
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 9)
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    
    _, _, q50_pred, _ = train_xgboost_quantiles(
        X_train, y_train, X_test, 
        learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators
    )
    return mean_absolute_error(y_test, q50_pred)


def objective_lgb(trial):
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    num_leaves = trial.suggest_int('num_leaves', 15, 127)
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    
    _, _, q50_pred, _ = train_lightgbm_quantiles(
        X_train, y_train, X_test, 
        learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators
    )
    return mean_absolute_error(y_test, q50_pred)


def objective_sklearn(trial):
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 9)
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    
    _, _, q50_pred, _ = train_sklearn_gb_quantiles(
        X_train, y_train, X_test, 
        learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators
    )
    return mean_absolute_error(y_test, q50_pred)


print("Avvio ottimizzazione iperparametri con Optuna...\n")
n_trials = 25  # Aumenta questo valore se desideri una ricerca più approfondita (es. 50 o 100)

# 1. XGBoost
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=n_trials)
print(f"Migliori parametri XGBoost: {study_xgb.best_params} | MAE: {study_xgb.best_value:.4f}")

# 2. LightGBM
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=n_trials)
print(f"Migliori parametri LightGBM: {study_lgb.best_params} | MAE: {study_lgb.best_value:.4f}")

# 3. Sklearn GB
study_sk = optuna.create_study(direction='minimize')
study_sk.optimize(objective_sklearn, n_trials=n_trials)
print(f"Migliori parametri Sklearn GB: {study_sk.best_params} | MAE: {study_sk.best_value:.4f}")


Avvio ottimizzazione iperparametri con Optuna...

Migliori parametri XGBoost: {'learning_rate': 0.016023369604557667, 'max_depth': 7, 'n_estimators': 252} | MAE: 0.3865
Migliori parametri LightGBM: {'learning_rate': 0.03603590590417038, 'num_leaves': 33, 'n_estimators': 226} | MAE: 0.3966
Migliori parametri Sklearn GB: {'learning_rate': 0.1985171413542642, 'max_depth': 3, 'n_estimators': 300} | MAE: 0.3753


In [ ]:
print("--- ADDESTRAMENTO E CONFRONTO MODELLI FINALI ---")

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Definiamo i dizionari per addestrare i modelli con i parametri ottimali trovati
models_optimized = {
    "XGBoost": lambda X_tr, y_tr, X_te: train_xgboost_quantiles(
        X_tr, y_tr, X_te, **study_xgb.best_params
    ),
    "LightGBM": lambda X_tr, y_tr, X_te: train_lightgbm_quantiles(
        X_tr, y_tr, X_te, **study_lgb.best_params
    ),
    "Sklearn GB": lambda X_tr, y_tr, X_te: train_sklearn_gb_quantiles(
        X_tr, y_tr, X_te, **study_sk.best_params
    ),
    "TabPFN": lambda X_tr, y_tr, X_te: train_tabpfn_quantiles(
        X_tr, y_tr, X_te, n_ensemble=12, device=device
    )
}

best_mae = float('inf')
best_model_name = ""
best_predictions = None
best_trained_models = None

for name, train_func in models_optimized.items():
    trained_models, q10, q50, q90 = train_func(X_train, y_train, X_test)
    mae = evaluate_model(name, y_test, q50, q10, q90)
    
    if mae < best_mae:
        best_mae = mae
        best_model_name = name
        best_predictions = (q10, q50, q90)
        best_trained_models = trained_models

print(f"Modello migliore selezionato: {best_model_name} con MAE di {best_mae:.3f}")

# Salvataggio su file del modello ottimizzato vincitore
os.makedirs("saved_models", exist_ok=True)
joblib.dump(best_trained_models, "saved_models/best_quantiles_model.pkl")
print("Modello salvato con successo in 'saved_models/best_quantiles_model.pkl'")

--- ADDESTRAMENTO E CONFRONTO MODELLI FINALI ---
XGBoost      | MAE: 0.386 | RMSE: 0.621 | R^2: 0.759 | Copertura: 89.7%
LightGBM     | MAE: 0.397 | RMSE: 0.627 | R^2: 0.755 | Copertura: 77.7%
Sklearn GB   | MAE: 0.375 | RMSE: 0.613 | R^2: 0.765 | Copertura: 92.4%
Modello migliore selezionato: Sklearn GB con MAE di 0.375
Modello salvato con successo in 'saved_models/best_quantiles_model.pkl'
